In [ ]:
import matplotlib.pyplot as plt
import cv2
import os
import numpy as np
import random
import tensorflow as tf
from keras_preprocessing import image
from keras_preprocessing.image import ImageDataGenerator
import hada as hd
from keras.datasets import mnist

#(train_X, train_y), (test_X, test_y) = mnist.load_data()


In [ ]:
current_path = os.getcwd()
train_dir = 'images/training'
test_dir = 'images/testing'

train_path = os.path.join(current_path, train_dir)
test_path = os.path.join(current_path, test_dir)

#os.chdir(train_path)

train_datagen = ImageDataGenerator(rescale = 1./255)
test_datagen = ImageDataGenerator(rescale = 1./255)

train_generator = train_datagen.flow_from_directory(
	train_path,
	target_size=(28,28),
	class_mode='categorical',
  batch_size=1345
)

test_generator = test_datagen.flow_from_directory(
	test_path,
	target_size=(28,28),
	class_mode='categorical',
  batch_size=141
)

In [ ]:
train_X,train_y = train_generator[0][0],train_generator[0][1]
test_X,test_y= test_generator[0][0],test_generator[0][1]

In [ ]:
data_size = train_X.shape[0]
image_size = (train_X.shape[1:3])
randomv = random.randint(0,data_size)
chars = [3,4,5,6]

In [ ]:
plt.imshow(train_X[0])
plt.show()
print(train_y[0])
train_X.shape , train_y.shape

In [ ]:
hady = cv2.resize(train_X[0],(1280,720))
plt.imshow(hady)
plt.show()

In [ ]:
rtrain = hd.Modify(train_X)
rtest = hd.Modify(test_X)

In [ ]:
Train_set = np.zeros((rtrain.shape[0],1280,720,3))
Test_set = np.zeros((rtest.shape[0],1280,720,3))
for i in range(rtrain.shape[0]):
        rtest[i]= hd.Resize(rtest[i])
for i in range(rtest.shape[0]):
        rtest[i]= hd.Resize(rtest[i])

In [ ]:
rtrain.shape, train_y.shape

In [ ]:
len(rtrain),rtrain[randomv].shape

In [ ]:
directory = "IMAGES"
path = os.path.join(current_path, directory)
if not os.path.exists(path):
        os.mkdir(path)

In [ ]:
for x in range(rtrain.shape[0]):
    filename = f"IMAGES/image_{x}.png"
    #cv2.imwrite(filename,rtrain[x])
    #cv2.waitKey(0)
    fig = plt.figure()
    ax = fig.add_subplot()
    plt.imshow(rtrain[x])
    #plt.show()
    fig.savefig(filename)

In [ ]:
plt.imshow(rtrain[random.randint(0,data_size)])
plt.show()

In [ ]:
model = tf.keras.models.Sequential()

model.add(tf.keras.layers.Conv2D(64,(3,3), activation= tf.nn.relu, input_shape=(1280,720,3)))
model.add(tf.keras.layers.MaxPool2D(2,2))
model.add(tf.keras.layers.Flatten(input_shape=(1280,720,3)))
model.add(tf.keras.layers.Dropout(0.3))
model.add(tf.keras.layers.Dense(units=64, activation= tf.nn.relu))
model.add(tf.keras.layers.Dropout(0.3))
model.add(tf.keras.layers.Dense(units=32, activation= tf.nn.relu))
model.add(tf.keras.layers.Dense(units=4, activation= tf.nn.softmax))

model.compile(optimizer= "adam", loss='categorical_crossentropy' , metrics= ["accuracy"])
model.summary()
model.fit(rtrain,train_y, epochs = 20, batch_size=16)



In [ ]:
loss,accuracy = model.evaluate(rtest, test_y)
print('accuracy =',accuracy)

model.save("digit.model")

In [ ]:
from sklearn.metrics import confusion_matrix

acc = model.predict(rtest)
for i in range(30):
    print(acc[i],test_y[i])
conf = confusion_matrix(rtest, test_y)
print(conf)